#Importing Libraries

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

#Reading Dataset

In [2]:
dataframe = pd.read_csv('../data/raw/credit_card_fraud_dataset.csv')

#Becoming One With Data

In [3]:
dataframe.head()

,TransactionID,TransactionDate,Amount,MerchantID,TransactionType,Location,IsFraud
0,1,03-04-2024,4189.27,688,refund,San Antonio,0
1,2,19-03-2024,2659.71,109,refund,Dallas,0
2,3,08-01-2024,784.00,394,purchase,New York,0
3,4,13-04-2024,3514.40,944,purchase,Philadelphia,0
4,5,12-07-2024,369.07,475,purchase,Phoenix,0


In [4]:
from sklearn.preprocessing import RobustScaler

# Initialize the RobustScaler
rob_scaler = RobustScaler()

# Scale the amount column to handle massive outliers
dataframe['scaled_amount'] = rob_scaler.fit_transform(dataframe['Amount'].values.reshape(-1,1))

# Drop the old unscaled amount column
dataframe.drop(['Amount'], axis=1, inplace=True)

# Look at the data to confirm it worked
dataframe.head()

,TransactionID,TransactionDate,MerchantID,TransactionType,Location,IsFraud,scaled_amount
0,1,03-04-2024,688,refund,San Antonio,0,0.678292
1,2,19-03-2024,109,refund,Dallas,0,0.065398
2,3,08-01-2024,394,purchase,New York,0,-0.686197
3,4,13-04-2024,944,purchase,Philadelphia,0,0.407872
4,5,12-07-2024,475,purchase,Phoenix,0,-0.852460


In [5]:
from sklearn.model_selection import train_test_split
import pandas as pd

# 1. Drop columns that don't help the ML model predict fraud
dataframe = dataframe.drop(['TransactionID', 'TransactionDate'], axis=1)

# 2. Convert text columns (TransactionType, Location) into numbers (1s and 0s)
dataframe = pd.get_dummies(dataframe, columns=['TransactionType', 'Location'], drop_first=True)

# 3. Separate features (X) from target label (y) using your actual column name
X = dataframe.drop('IsFraud', axis=1) 
y = dataframe['IsFraud']

# 4. Split 80% of data for training, 20% for testing
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print("Training data shape:", X_train.shape)
print("Testing data shape:", X_test.shape)

Training data shape: (80000, 12)
Testing data shape: (20000, 12)


In [6]:
%pip install imbalanced-learn

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.1.1
[notice] To update, run: C:\Users\mrsha\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [7]:
from imblearn.over_sampling import SMOTE

print("Before SMOTE, fraud cases in train set:", sum(y_train == 1))

# Apply SMOTE to the training data ONLY
smote = SMOTE(random_state=42)
X_train_smote, y_train_smote = smote.fit_resample(X_train, y_train)

print("After SMOTE, fraud cases in train set:", sum(y_train_smote == 1))

Before SMOTE, fraud cases in train set: 800
After SMOTE, fraud cases in train set: 79200


In [8]:
# Save the cleaned and balanced data into our structured folders
X_train_smote.to_csv('../data/processed/X_train_smote.csv', index=False)
y_train_smote.to_csv('../data/processed/y_train_smote.csv', index=False)
X_test.to_csv('../data/processed/X_test.csv', index=False)
y_test.to_csv('../data/processed/y_test.csv', index=False)

print("Data successfully saved to data/processed folder!")

Data successfully saved to data/processed folder!
